In [ ]:
"""
=============================================================================
DS 4320 - HW8 Section 8.2: Financial Data Engineering & Forecasting Pipeline
=============================================================================
GOAL:    1 GB+ Multi-Table Analytical Database
STACK:   Python 3.x, DuckDB, yfinance, Parquet
TABLES:  Companies, PriceHistory, Fundamentals, TechnicalIndicators, StockNews
=============================================================================

PROVENANCE NOTE:
    All financial data is sourced in real-time from:
    - Yahoo Finance via the `yfinance` library (price history, fundamentals, metadata)
    - Wikipedia S&P 500 constituent list (ticker universe)
    - Yahoo Finance RSS feeds (news headlines for sentiment analysis)
    No synthetic or fake data is used. Data is pulled on execution date.
=============================================================================
"""

# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 – Install Dependencies
# ─────────────────────────────────────────────────────────────────────────────
# Run this cell first in Google Colab

INSTALL_CELL = """
!pip install -q yfinance duckdb pandas pyarrow requests lxml
"""

# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 – Imports & Configuration
# ─────────────────────────────────────────────────────────────────────────────

import yfinance as yf
import duckdb
import pandas as pd
import numpy as np
import requests
import time
import os
import gc
import re
import xml.etree.ElementTree as ET
from datetime import datetime, date
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────────────────
DB_PATH       = "stock_data.db"
PARQUET_DIR   = Path("parquet_exports")
PARQUET_DIR.mkdir(exist_ok=True)

# Throttling: be polite to Yahoo Finance to avoid rate-limiting / 429 errors
REQUEST_DELAY = 0.4   # seconds between ticker requests
BATCH_SIZE    = 50    # flush to DB every N tickers (keeps RAM low)

# Max tickers — use 500 for a real 1 GB run; reduce for testing
MAX_TICKERS   = 500   # set to e.g. 20 for a quick test run

print(f"[config] DB={DB_PATH}  parquet={PARQUET_DIR}  max_tickers={MAX_TICKERS}")

# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 – Ticker Universe: S&P 500 from Wikipedia
# ─────────────────────────────────────────────────────────────────────────────

def get_sp500_tickers() -> list[str]:
    """
    Scrapes the S&P 500 constituent list from Wikipedia.

    WHY requests INSTEAD OF pd.read_html(url) DIRECTLY?
    ─────────────────────────────────────────────────────
    pandas' read_html() fetches URLs using Python's stdlib urllib, which sends
    a bare 'Python-urllib/3.x' User-Agent string.  Wikipedia's CDN (Wikimedia)
    blocks this with a 403 Forbidden.  The fix is to fetch the page ourselves
    using requests (which sends a realistic browser UA), then pass the raw HTML
    string — not the URL — into pd.read_html().  pd.read_html accepts a string
    of HTML directly and never touches the network when given one.

    Returns
    -------
    list[str]  – deduplicated, sorted list of ticker symbols
    """
    url     = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()

    # Wrap in StringIO: newer pandas treats raw HTML strings as file paths,
    # causing FileNotFoundError. StringIO makes it a file-like object instead.
    from io import StringIO
    tables = pd.read_html(StringIO(resp.text), attrs={"id": "constituents"})
    df = tables[0]
    # Wikipedia uses 'Symbol' column; dots replaced with hyphens for yfinance
    tickers = (df["Symbol"]
               .str.replace(".", "-", regex=False)
               .dropna()
               .unique()
               .tolist())
    tickers.sort()
    print(f"[tickers] Loaded {len(tickers)} S&P 500 constituents from Wikipedia")
    return tickers[:MAX_TICKERS]


# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 – DuckDB Schema Creation
# ─────────────────────────────────────────────────────────────────────────────

def create_schema(con: duckdb.DuckDBPyConnection) -> None:
    """
    Creates all five tables in DuckDB if they do not already exist.

    Design Decisions / Rationale:
    ─────────────────────────────
    • Companies     : one row per ticker; longBusinessSummary adds string bulk
                      (intentional for size target).
    • PriceHistory  : Volume is BIGINT (INT64) to handle NVDA-class stocks whose
                      daily volume routinely exceeds 2^31 (~2.1 B shares), which
                      overflows a 32-bit integer.
    • Fundamentals  : Long/EAV format (symbol, report_date, metric, value)
                      avoids the 'catalog error' caused by wide tables where
                      different industries report different metric sets.
    • TechnicalIndicators: Wide format is intentional here — each row is a
                      (symbol, date) pair that directly feeds ML feature matrices.
    • StockNews     : Raw text table; bulk comes from title + summary strings.
                      Suitable for NLP / sentiment analysis tasks.
    """
    con.execute("""
        CREATE TABLE IF NOT EXISTS Companies (
            symbol              VARCHAR PRIMARY KEY,
            short_name          VARCHAR,
            long_name           VARCHAR,
            sector              VARCHAR,
            industry            VARCHAR,
            country             VARCHAR,
            exchange            VARCHAR,
            market_cap          BIGINT,
            full_time_employees INTEGER,
            website             VARCHAR,
            long_business_summary TEXT,    -- intentionally large text field
            fetched_at          TIMESTAMP DEFAULT current_timestamp
        )
    """)

    con.execute("""
        CREATE TABLE IF NOT EXISTS PriceHistory (
            symbol   VARCHAR      NOT NULL,
            date     DATE         NOT NULL,
            open     DOUBLE,
            high     DOUBLE,
            low      DOUBLE,
            close    DOUBLE,
            volume   BIGINT,          -- MUST be BIGINT — avoids int32 overflow
            adj_close DOUBLE,
            PRIMARY KEY (symbol, date)
        )
    """)

    con.execute("""
        CREATE TABLE IF NOT EXISTS Fundamentals (
            symbol      VARCHAR  NOT NULL,
            report_date DATE,
            period_type VARCHAR,         -- 'annual' or 'quarterly'
            metric      VARCHAR  NOT NULL,
            value       DOUBLE,
            PRIMARY KEY (symbol, report_date, period_type, metric)
        )
    """)

    con.execute("""
        CREATE TABLE IF NOT EXISTS TechnicalIndicators (
            symbol      VARCHAR NOT NULL,
            date        DATE    NOT NULL,
            close       DOUBLE,
            -- Returns
            daily_return        DOUBLE,
            log_return          DOUBLE,
            cumulative_return   DOUBLE,
            -- Simple Moving Averages
            sma_5    DOUBLE,
            sma_10   DOUBLE,
            sma_20   DOUBLE,
            sma_50   DOUBLE,
            sma_200  DOUBLE,
            -- Exponential Moving Averages
            ema_12   DOUBLE,
            ema_26   DOUBLE,
            ema_50   DOUBLE,
            -- MACD
            macd            DOUBLE,
            macd_signal     DOUBLE,
            macd_histogram  DOUBLE,
            -- Bollinger Bands (20-day, 2 std)
            bb_upper DOUBLE,
            bb_middle DOUBLE,
            bb_lower DOUBLE,
            bb_width DOUBLE,
            bb_pct_b DOUBLE,
            -- RSI (14-day)
            rsi_14   DOUBLE,
            -- Volume indicators
            volume          BIGINT,
            volume_sma_20   DOUBLE,
            volume_ratio    DOUBLE,     -- volume / volume_sma_20
            -- Volatility
            atr_14          DOUBLE,     -- Average True Range
            hist_vol_20     DOUBLE,     -- 20-day historical volatility (annualised)
            PRIMARY KEY (symbol, date)
        )
    """)

    con.execute("""
        CREATE TABLE IF NOT EXISTS StockNews (
            id          INTEGER PRIMARY KEY,
            symbol      VARCHAR,
            title       VARCHAR,
            publisher   VARCHAR,
            link        VARCHAR,
            provider_publish_time TIMESTAMP,
            news_type   VARCHAR,
            fetched_at  TIMESTAMP DEFAULT current_timestamp
        )
    """)

    print("[schema] All tables created / verified.")


# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 – Technical Indicator Calculations (Pure Pandas — no Numba dependency)
# ─────────────────────────────────────────────────────────────────────────────

def compute_technical_indicators(hist: pd.DataFrame, symbol: str) -> pd.DataFrame:
    """
    Computes ~20 technical features using only pandas .rolling() and .ewm().

    WHY PURE PANDAS?
    ────────────────
    Libraries like pandas_ta and TA-Lib depend on Numba or C extensions that
    are not yet compatible with Python 3.14's updated ABI.  All indicators
    here are implemented from first principles via pandas built-ins, making
    the pipeline version-safe.

    Parameters
    ----------
    hist   : pd.DataFrame  – yfinance price history (Date index, OHLCV columns)
    symbol : str

    Returns
    -------
    pd.DataFrame with one row per trading day and all indicator columns.
    """
    if hist.empty or len(hist) < 30:
        return pd.DataFrame()

    df = hist.copy()
    df.index = pd.to_datetime(df.index).normalize()   # strip tz, keep date
    df.index.name = "date"

    # ── Returns ──────────────────────────────────────────────────────────────
    df["daily_return"]      = df["Close"].pct_change()
    df["log_return"]        = np.log(df["Close"] / df["Close"].shift(1))
    df["cumulative_return"] = (1 + df["daily_return"]).cumprod() - 1

    # ── Simple Moving Averages ────────────────────────────────────────────────
    for w in [5, 10, 20, 50, 200]:
        df[f"sma_{w}"] = df["Close"].rolling(w).mean()

    # ── Exponential Moving Averages ───────────────────────────────────────────
    for span in [12, 26, 50]:
        df[f"ema_{span}"] = df["Close"].ewm(span=span, adjust=False).mean()

    # ── MACD ─────────────────────────────────────────────────────────────────
    df["macd"]           = df["ema_12"] - df["ema_26"]
    df["macd_signal"]    = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_histogram"] = df["macd"] - df["macd_signal"]

    # ── Bollinger Bands (20-day, 2σ) ─────────────────────────────────────────
    roll20       = df["Close"].rolling(20)
    df["bb_middle"] = roll20.mean()
    bb_std          = roll20.std()
    df["bb_upper"]  = df["bb_middle"] + 2 * bb_std
    df["bb_lower"]  = df["bb_middle"] - 2 * bb_std
    df["bb_width"]  = (df["bb_upper"] - df["bb_lower"]) / df["bb_middle"]
    df["bb_pct_b"]  = (df["Close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])

    # ── RSI (14-day, Wilder smoothing via ewm) ────────────────────────────────
    delta    = df["Close"].diff()
    gain     = delta.clip(lower=0)
    loss     = (-delta).clip(lower=0)
    avg_gain = gain.ewm(com=13, adjust=False).mean()
    avg_loss = loss.ewm(com=13, adjust=False).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    df["rsi_14"] = 100 - (100 / (1 + rs))

    # ── Volume Indicators ─────────────────────────────────────────────────────
    df["volume_sma_20"] = df["Volume"].rolling(20).mean()
    df["volume_ratio"]  = df["Volume"] / df["volume_sma_20"]

    # ── Average True Range (14-day) ───────────────────────────────────────────
    high_low   = df["High"] - df["Low"]
    high_prev  = (df["High"] - df["Close"].shift(1)).abs()
    low_prev   = (df["Low"]  - df["Close"].shift(1)).abs()
    true_range = pd.concat([high_low, high_prev, low_prev], axis=1).max(axis=1)
    df["atr_14"] = true_range.ewm(com=13, adjust=False).mean()

    # ── Historical Volatility (annualised, 20-day window) ─────────────────────
    df["hist_vol_20"] = df["log_return"].rolling(20).std() * np.sqrt(252)

    # ── Tidy & Return ─────────────────────────────────────────────────────────
    df["symbol"] = symbol
    df["volume"] = df["Volume"].fillna(0).astype(np.int64) 
    df = df.rename(columns={"Close": "close"})

    keep = [
        "symbol", "close",
        "daily_return", "log_return", "cumulative_return",
        "sma_5", "sma_10", "sma_20", "sma_50", "sma_200",
        "ema_12", "ema_26", "ema_50",
        "macd", "macd_signal", "macd_histogram",
        "bb_upper", "bb_middle", "bb_lower", "bb_width", "bb_pct_b",
        "rsi_14",
        "volume", "volume_sma_20", "volume_ratio",
        "atr_14", "hist_vol_20",
    ]
    result = df[keep].reset_index()
    result["date"] = result["date"].dt.date
    return result


# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 – Fundamentals: Wide → Long (EAV) Transformation
# ─────────────────────────────────────────────────────────────────────────────

def extract_fundamentals(ticker_obj: yf.Ticker, symbol: str) -> pd.DataFrame:
    """
    Pulls income statement, balance sheet, and cash-flow data from yfinance
    and stacks them into long (EAV) format.

    WHY LONG FORMAT?
    ────────────────
    Different industries report different line items.  A wide table would have
    hundreds of mostly-null columns ('Catalog Errors' in DuckDB parlance when
    column counts change mid-load).  The EAV pattern gives a stable 4-column
    schema regardless of which metrics each company reports.

    UNCERTAINTY NOTE:
    ─────────────────
    yfinance sourcing is best-effort; some tickers return empty frames.
    The period_type column records whether data is annual or quarterly so
    downstream models can filter correctly.
    """
    records = []

    def _stack(df: pd.DataFrame, period_type: str):
        if df is None or df.empty:
            return
        # Columns are report dates, rows are metrics
        for col in df.columns:
            for metric, val in df[col].items():
                if pd.notna(val):
                    records.append({
                        "symbol":      symbol,
                        "report_date": col.date() if hasattr(col, 'date') else col,
                        "period_type": period_type,
                        "metric":      str(metric),
                        "value":       float(val),
                    })

    try:
        _stack(ticker_obj.income_stmt,            "annual")
        _stack(ticker_obj.quarterly_income_stmt,  "quarterly")
        _stack(ticker_obj.balance_sheet,          "annual")
        _stack(ticker_obj.quarterly_balance_sheet,"quarterly")
        _stack(ticker_obj.cash_flow,              "annual")
        _stack(ticker_obj.quarterly_cash_flow,    "quarterly")
    except Exception as e:
        print(f"    [fundamentals] Warning for {symbol}: {e}")

    return pd.DataFrame(records) if records else pd.DataFrame()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 – News via Yahoo Finance RSS (real unstructured data)
# ─────────────────────────────────────────────────────────────────────────────

def fetch_news_rss(symbol: str, max_items: int = 30) -> pd.DataFrame:
    """
    Fetches real news headlines from Yahoo Finance's public RSS feed.

    This is genuine unstructured text data — no fabrication.  The RSS feed
    is public and does not require authentication.  Text volume from titles
    and publisher names contributes meaningfully toward the 1 GB target.

    BIAS NOTE:
    ──────────
    RSS feeds are recency-biased (only the last ~30 items) and English-language
    only, which over-represents US/English-speaking market events.

    Parameters
    ----------
    symbol    : ticker symbol
    max_items : cap per ticker (default 30)
    """
    url = f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={symbol}&region=US&lang=en-US"
    records = []
    try:
        resp = requests.get(url, timeout=10,
                            headers={"User-Agent": "Mozilla/5.0 (research bot)"})
        if resp.status_code != 200:
            return pd.DataFrame()

        root = ET.fromstring(resp.content)
        ns   = {"media": "http://search.yahoo.com/mrss/"}
        items = root.findall(".//item")[:max_items]

        for item in items:
            title = item.findtext("title", "").strip()
            link  = item.findtext("link",  "").strip()
            pub   = item.findtext("pubDate", "")
            src   = item.findtext("source", "Yahoo Finance")
            # Parse publish time
            try:
                pt = datetime.strptime(pub, "%a, %d %b %Y %H:%M:%S %z")
            except Exception:
                pt = None
            records.append({
                "symbol":                symbol,
                "title":                 title,
                "publisher":             src,
                "link":                  link,
                "provider_publish_time": pt,
                "news_type":             "RSS",
            })
    except Exception as e:
        pass  # network hiccup — skip gracefully

    return pd.DataFrame(records) if records else pd.DataFrame()


# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 – Main Extraction Loop
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline():
    """
    Full ETL loop:  Extract → Transform → Load → Export

    IMPLEMENTATION NOTES (Critical Technical Hurdles addressed):
    ────────────────────────────────────────────────────────────
    1. Python 3.14 Compatibility: All TA computed with pandas rolling/ewm only.
    2. Integer Overflow: Volume cast to Int64 (BIGINT in DuckDB) everywhere.
    3. Schema Stability: Fundamentals in EAV long format.
    4. Rate Limiting: REQUEST_DELAY seconds between tickers; batch flushing
       keeps RAM under control and allows resumption after interruption.
    5. Ticker Validity: We catch all exceptions per-ticker so one bad symbol
       cannot abort the entire run.
    """

    tickers = get_sp500_tickers()
    con     = duckdb.connect(DB_PATH)
    con.execute("PRAGMA memory_limit='12GB'")

    for table in ["Companies", "PriceHistory", "Fundamentals", "TechnicalIndicators", "StockNews"]:
        con.execute(f"DROP TABLE IF EXISTS {table}")
    
    create_schema(con)

    # ── Batch accumulators ────────────────────────────────────────────────────
    co_batch   = []   # Companies
    ph_batch   = []   # PriceHistory
    fu_batch   = []   # Fundamentals
    ti_batch   = []   # TechnicalIndicators
    sn_batch   = []   # StockNews
    news_id    = 0

    def flush():
        """Append accumulated DataFrames to DuckDB tables."""
        nonlocal news_id
        if co_batch:
            df_co = pd.DataFrame(co_batch)
            con.execute("""
                INSERT INTO Companies (symbol, short_name, long_name, sector, industry, country, exchange, market_cap, full_time_employees, website, long_business_summary)
                SELECT symbol, short_name, long_name, sector, industry, country, exchange, market_cap, full_time_employees, website, long_business_summary FROM df_co
                ON CONFLICT (symbol) DO UPDATE SET
                    short_name = EXCLUDED.short_name,
                    long_name = EXCLUDED.long_name,
                    sector = EXCLUDED.sector,
                    industry = EXCLUDED.industry,
                    country = EXCLUDED.country,
                    exchange = EXCLUDED.exchange,
                    market_cap = EXCLUDED.market_cap,
                    full_time_employees = EXCLUDED.full_time_employees,
                    website = EXCLUDED.website,
                    long_business_summary = EXCLUDED.long_business_summary,
                    fetched_at = now()
            """)
            co_batch.clear()

        if ph_batch:
            df = pd.concat(ph_batch, ignore_index=True)
            con.execute("""
                INSERT INTO PriceHistory (symbol, date, open, high, low, close, volume, adj_close) 
                SELECT symbol, date, open, high, low, close, volume, adj_close FROM df
                ON CONFLICT (symbol, date) DO UPDATE SET
                    open = EXCLUDED.open, high = EXCLUDED.high, low = EXCLUDED.low, 
                    close = EXCLUDED.close, volume = EXCLUDED.volume, adj_close = EXCLUDED.adj_close
            """)
            ph_batch.clear()

        if fu_batch:
            df = pd.concat(fu_batch, ignore_index=True)
            con.execute("""
                INSERT INTO Fundamentals (symbol, report_date, period_type, metric, value) 
                SELECT symbol, report_date, period_type, metric, value FROM df
                ON CONFLICT (symbol, report_date, period_type, metric) DO UPDATE SET
                    value = EXCLUDED.value
            """)
            fu_batch.clear()

        if ti_batch:
            df = pd.concat(ti_batch, ignore_index=True)
            cols = ['symbol', 'date', 'close', 'daily_return', 'log_return', 'cumulative_return', 'sma_5', 'sma_10', 'sma_20', 'sma_50', 'sma_200', 'ema_12', 'ema_26', 'ema_50', 'macd', 'macd_signal', 'macd_histogram', 'bb_upper', 'bb_middle', 'bb_lower', 'bb_width', 'bb_pct_b', 'rsi_14', 'volume', 'volume_sma_20', 'volume_ratio', 'atr_14', 'hist_vol_20']
            c_str = ", ".join(cols)
            set_str = ", ".join([f"{c} = EXCLUDED.{c}" for c in cols if c not in ('symbol', 'date')])
            con.execute(f"INSERT INTO TechnicalIndicators ({c_str}) SELECT {c_str} FROM df ON CONFLICT (symbol, date) DO UPDATE SET {set_str}")
            ti_batch.clear()

        if sn_batch:
            df = pd.concat(sn_batch, ignore_index=True)
            df.insert(0, "id", range(news_id, news_id + len(df)))
            news_id += len(df)
            con.execute("INSERT OR IGNORE INTO StockNews (id, symbol, title, publisher, link, provider_publish_time, news_type) SELECT id, symbol, title, publisher, link, provider_publish_time, news_type FROM df")
            sn_batch.clear()

        gc.collect()

        # Safely calculate size of both DB and WAL files
        db_size = os.path.getsize(DB_PATH) if os.path.exists(DB_PATH) else 0
        wal_size = os.path.getsize(DB_PATH + ".wal") if os.path.exists(DB_PATH + ".wal") else 0
        total_mb = (db_size + wal_size) / 1e6
        
        print(f"    [flush] Storage size (DB + WAL): {total_mb:.1f} MB")

        con.execute("CHECKPOINT")

    # ── Per-Ticker Loop ───────────────────────────────────────────────────────
    for i, symbol in enumerate(tickers):
        print(f"[{i+1:>4}/{len(tickers)}] {symbol}", end=" ... ", flush=True)
        try:
            tk   = yf.Ticker(symbol)
            info = tk.info or {}

            # ── 1. Companies ─────────────────────────────────────────────────
            co_batch.append({
                "symbol":               symbol,
                "short_name":           info.get("shortName"),
                "long_name":            info.get("longName"),
                "sector":               info.get("sector"),
                "industry":             info.get("industry"),
                "country":              info.get("country"),
                "exchange":             info.get("exchange"),
                "market_cap":           info.get("marketCap"),
                "full_time_employees":  info.get("fullTimeEmployees"),
                "website":              info.get("website"),
                "long_business_summary":info.get("longBusinessSummary"),  # fat text
            })

            # ── 2. PriceHistory ──────────────────────────────────────────────
            hist = tk.history(period="max", auto_adjust=True)
            if not hist.empty:
                ph = hist[["Open","High","Low","Close","Volume"]].copy()
                ph.index = pd.to_datetime(ph.index).normalize()
                ph.index.name = "date"
                ph = ph.reset_index()
                ph["symbol"]    = symbol
                ph["date"]      = ph["date"].dt.date
                ph["volume"]    = ph["Volume"].fillna(0).astype("Int64")
                ph["adj_close"] = ph["Close"]   # auto_adjust=True → Close=Adj
                ph = ph.rename(columns={
                    "Open":"open","High":"high","Low":"low","Close":"close"
                })[["symbol","date","open","high","low","close","volume","adj_close"]]
                ph_batch.append(ph)

                # ── 3. TechnicalIndicators ───────────────────────────────────
                ti = compute_technical_indicators(hist, symbol)
                if not ti.empty:
                    ti_batch.append(ti)

            # ── 4. Fundamentals ──────────────────────────────────────────────
            fu = extract_fundamentals(tk, symbol)
            if not fu.empty:
                fu_batch.append(fu)

            # ── 5. StockNews (RSS) ───────────────────────────────────────────
            sn = fetch_news_rss(symbol, max_items=30)
            if not sn.empty:
                sn_batch.append(sn)

            print("✓")

        except Exception as e:
            print(f"✗  [{type(e).__name__}] {e}")

        if 'tk' in locals(): del tk
        if 'hist' in locals(): del hist
        if 'info' in locals(): del info
        gc.collect()

        time.sleep(REQUEST_DELAY)

        # Flush every BATCH_SIZE tickers
        if (i + 1) % BATCH_SIZE == 0:
            print(f"\n─── Flushing batch {(i+1)//BATCH_SIZE} ───")
            flush()

    # Final flush
    print("\n─── Final flush ───")
    flush()

    # ── Export to Parquet ────────────────────────────────────────────────────
    print("\n[export] Writing Parquet files via DuckDB COPY engine ...")
    for table in ["Companies", "PriceHistory", "Fundamentals",
                  "TechnicalIndicators", "StockNews"]:
        out = PARQUET_DIR / f"{table}.parquet"
        con.execute(f"COPY {table} TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
        size_mb = out.stat().st_size / 1e6
        print(f"  → {out}  ({size_mb:.1f} MB)")

    # ── Verification Queries ─────────────────────────────────────────────────
    print("\n═══════════════════════════════════════════")
    print("  VERIFICATION QUERIES")
    print("═══════════════════════════════════════════")

    queries = {
        "PriceHistory row count":
            "SELECT COUNT(*) AS rows FROM PriceHistory",
        "TechnicalIndicators row count":
            "SELECT COUNT(*) AS rows FROM TechnicalIndicators",
        "Fundamentals row count":
            "SELECT COUNT(*) AS rows FROM Fundamentals",
        "StockNews row count":
            "SELECT COUNT(*) AS rows FROM StockNews",
        "Companies row count":
            "SELECT COUNT(*) AS rows FROM Companies",
        "AAPL TechnicalIndicators sample":
            "SELECT * FROM TechnicalIndicators WHERE symbol='AAPL' LIMIT 5",
        "Sector distribution":
            "SELECT sector, COUNT(*) AS n FROM Companies GROUP BY sector ORDER BY n DESC",
        "Date range":
            "SELECT MIN(date), MAX(date) FROM PriceHistory",
    }

    for label, sql in queries.items():
        print(f"\n── {label}")
        print(con.execute(sql).df().to_string(index=False))

    db_mb = os.path.getsize(DB_PATH) / 1e6
    print(f"\n[done] stock_data.db size: {db_mb:.1f} MB")
    con.close()


if __name__ == "__main__":
    run_pipeline()

[config] DB=stock_data.db  parquet=parquet_exports  max_tickers=500
[tickers] Loaded 503 S&P 500 constituents from Wikipedia
[schema] All tables created / verified.
[   1/500] A ... ✓
[   2/500] AAPL ... ✓
[   3/500] ABBV ... ✓
[   4/500] ABNB ... ✓
[   5/500] ABT ... ✓
[   6/500] ACGL ... ✓
[   7/500] ACN ... ✓
[   8/500] ADBE ... ✓
[   9/500] ADI ... ✓
[  10/500] ADM ... ✓
[  11/500] ADP ... ✓
[  12/500] ADSK ... ✓
[  13/500] AEE ... ✓
[  14/500] AEP ... ✓
[  15/500] AES ... ✓
[  16/500] AFL ... ✓
[  17/500] AIG ... ✓
[  18/500] AIZ ... ✓
[  19/500] AJG ... ✓
[  20/500] AKAM ... ✓
[  21/500] ALB ... ✓
[  22/500] ALGN ... ✓
[  23/500] ALL ... ✓
[  24/500] ALLE ... ✓
[  25/500] AMAT ... ✓
[  26/500] AMCR ... ✓
[  27/500] AMD ... ✓
[  28/500] AME ... ✓
[  29/500] AMGN ... ✓
[  30/500] AMP ... ✓
[  31/500] AMT ... ✓
[  32/500] AMZN ... ✓
[  33/500] ANET ... ✓
[  34/500] AON ... ✓
[  35/500] AOS ... ✓
[  36/500] APA ... ✓
[  37/500] APD ... ✓
[  38/500] APH ... ✓
[  39/500] APO ... ✓
[  4

: 